# Test: ParticleSpringForceDiffusionField generator

Visualize 3D particles with spring forces + chemotaxis from a diffusion-decay field with cell sources.

PDE: dc/dt = D * laplacian(c) - lambda * c + alpha * sum_i delta(x - x_i)

Three panels per frame:
1. 3D scatter of particle positions
2. Top-down (xy) projection of particles colored by local field value
3. Top-down field slice (z=0.5) with particles overlaid

In [13]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import trange
from IPython.display import Video, display
import matplotlib.animation as animation

from cell_gnn.cell_state import CellState
from cell_gnn.utils import edges_radius_blockwise, choose_boundary_values
from cell_gnn.generators.particle_spring_force_diffusion_field import (
    ParticleSpringForceDiffusionField,
)
from cell_gnn.integrators import rk4_step

## Parameters

In [14]:
device = 'cpu'

# Parameters from config/misc/dicty_spring_force_rk4_diffusion_field_v1.yaml
n_cells = 1000
dimension = 3
n_frames = 8000
delta_t = 0.002
max_radius = 0.075
min_radius = 0.0
noise_level = 0.0

# spring force params: [k_rep, r0, kadh, r_on, delta, mu_f]
cell_params = torch.tensor([50.0, 0.05, 50.0, 0.07, 0.001, 0.033], device=device)

# diffusion field params
grid_resolution = 100

field_params = {
    'diffusion_coeff': 0.01,     # D
    'lambda_decay': 0.1,         # lambda
    'grid_resolution': grid_resolution,
    'source_strength': 20.0,      # alpha
    'center_0': torch.tensor([
        [0.50, 0.50, 0.50],
    ], device=device),
    'amplitude': 0.0,
    'sigma': 0.1,
    'mu_chem': 0.1,
    'delta_t': delta_t,
    'periodic': True,
}

# boundary conditions
bc_pos, bc_dpos = choose_boundary_values('periodic')

# Steady-state estimate: alpha * n_cells * dt / (lambda * res^3)
ss_est = field_params['source_strength'] * n_cells * delta_t / (field_params['lambda_decay'] * grid_resolution**3)
print(f"D = {field_params['diffusion_coeff']}, lambda = {field_params['lambda_decay']}")
print(f"source_strength = {field_params['source_strength']}")
print(f"grid_resolution = {grid_resolution}^3 = {grid_resolution**3} points")
print(f"mu_chem = {field_params['mu_chem']}")
print(f"max_radius = {max_radius}, cell_params = {cell_params.tolist()}")
print(f"Estimated steady-state per-cell contribution: {ss_est:.6f}")

D = 0.01, lambda = 0.1
source_strength = 20.0
grid_resolution = 100^3 = 1000000 points
mu_chem = 0.1
max_radius = 0.075, cell_params = [50.0, 0.05000000074505806, 50.0, 0.07000000029802322, 0.0010000000474974513, 0.032999999821186066]
Estimated steady-state per-cell contribution: 0.000400


## Initialize cells & model

In [15]:
torch.manual_seed(42)

# random initial positions in [0,1]^3
pos = torch.rand(n_cells, dimension, device=device)
vel = torch.zeros(n_cells, dimension, device=device)

x = CellState(
    index=torch.arange(n_cells, device=device),
    pos=pos,
    vel=vel,
    cell_type=torch.zeros(n_cells, dtype=torch.long, device=device),
    field=torch.ones(n_cells, 1, device=device),
)

model = ParticleSpringForceDiffusionField(
    aggr_type='add',
    p=cell_params,
    bc_dpos=bc_dpos,
    dimension=dimension,
    noise_model_level=noise_level,
    field_params=field_params,
)

print(f'Cells: {n_cells}, Frames: {n_frames}, dt: {delta_t}')

Cells: 1000, Frames: 8000, dt: 0.002


In [16]:
import types

source_fraction = 0.5
pulse_period = 500
pulse_duty = 0.5

torch.manual_seed(123)
n_src = max(1, int(round(source_fraction * n_cells)))
perm = torch.randperm(n_cells, device=device)
source_mask = torch.zeros(n_cells, dtype=torch.bool, device=device)
source_mask[perm[:n_src]] = True
source_phase = torch.rand(n_cells, device=device)

print(f"sources: {n_src}/{n_cells}, period={pulse_period} steps, duty={pulse_duty}")


def _deposit_weighted(pos, strength, resolution, dimension, dev):
    """Per-particle weighted trilinear/bilinear deposit onto a periodic grid."""
    grid_shape = [resolution] * dimension
    S = torch.zeros(grid_shape, device=dev)
    scaled = pos * resolution
    idx0 = scaled.long() % resolution
    idx1 = (idx0 + 1) % resolution
    frac = scaled - scaled.floor()

    if dimension == 3:
        ix0, iy0, iz0 = idx0[:, 0], idx0[:, 1], idx0[:, 2]
        ix1, iy1, iz1 = idx1[:, 0], idx1[:, 1], idx1[:, 2]
        fx, fy, fz = frac[:, 0], frac[:, 1], frac[:, 2]
        weights = [
            (iz0, iy0, ix0, (1 - fx) * (1 - fy) * (1 - fz)),
            (iz0, iy0, ix1, fx       * (1 - fy) * (1 - fz)),
            (iz0, iy1, ix0, (1 - fx) * fy       * (1 - fz)),
            (iz0, iy1, ix1, fx       * fy       * (1 - fz)),
            (iz1, iy0, ix0, (1 - fx) * (1 - fy) * fz),
            (iz1, iy0, ix1, fx       * (1 - fy) * fz),
            (iz1, iy1, ix0, (1 - fx) * fy       * fz),
            (iz1, iy1, ix1, fx       * fy       * fz),
        ]
        for iz, iy, ix, w in weights:
            S.index_put_((iz, iy, ix), strength * w, accumulate=True)
    else:
        ix0, iy0 = idx0[:, 0], idx0[:, 1]
        ix1, iy1 = idx1[:, 0], idx1[:, 1]
        fx, fy = frac[:, 0], frac[:, 1]
        weights = [
            (iy0, ix0, (1 - fx) * (1 - fy)),
            (iy0, ix1, fx       * (1 - fy)),
            (iy1, ix0, (1 - fx) * fy),
            (iy1, ix1, fx       * fy),
        ]
        for iy, ix, w in weights:
            S.index_put_((iy, ix), strength * w, accumulate=True)
    return S


def _patched_advance_field(self, pos, n_steps=1):
    dim = self.dimension
    res = self.field_params['grid_resolution']
    dt = self.field_params['delta_t']
    alpha = self.field_params.get('source_strength', 0.0)
    s = [res] * dim

    sigma = max(pulse_duty, 1e-3) / 2.0
    gain = 1.0 / max(pulse_duty, 1e-3)

    for i in range(n_steps):
        current_step = self._last_step + 1 + i
        if alpha > 0:
            phase = (current_step / float(pulse_period) + source_phase) % 1.0
            d = torch.minimum(phase, 1.0 - phase)
            pulse = torch.exp(-0.5 * (d / sigma) ** 2) * source_mask.float()
            active = pulse > 1e-4
            if active.any():
                strength = alpha * dt * gain * pulse[active]
                src = _deposit_weighted(pos[active], strength, res, dim, pos.device)
                self._field_grid = self._field_grid + src

        C_hat = torch.fft.rfftn(self._field_grid, s=s)
        C_hat = C_hat * self._decay
        self._field_grid = torch.fft.irfftn(C_hat, s=s)

    C_hat = torch.fft.rfftn(self._field_grid, s=s)
    return C_hat


model._advance_field = types.MethodType(_patched_advance_field, model)
print("patched model._advance_field with pulsed 20% sources")


sources: 500/1000, period=500 steps, duty=0.5
patched model._advance_field with pulsed 20% sources


In [17]:
# --- Weber-Fechner saturating chemotaxis (v2-style) ---
# F_chem = mu_chem * log(1 + |grad c| / g0) * (grad c / |grad c|)
#
# This cell defines `apply_wf_saturation(model)` which: warms up the field, measures
# pair/grad force magnitudes, picks g0 so saturating chem force matches pair force
# at the median, then patches model.forward to use the saturating form.
#
# Call it BEFORE running the sim if you want the v2 (saturation) data;
# skip it for v1 (linear chemotaxis) data.

from cell_gnn.generators.particle_spring_force_diffusion_field import (
    spectral_gradient, interp_periodic,
)
from torch_geometric.utils import remove_self_loops
from cell_gnn.generators.particle_spring_force_diffusion_field import scatter_aggregate
from copy import deepcopy


def apply_wf_saturation(model, x_init, warmup_steps=500):
    """Patch model.forward in-place to use Weber-Fechner log saturation.
    
    Returns the chosen g0 for diagnostics.
    """
    # --- 1. Warm-up
    x_probe = deepcopy(x_init)
    for it in range(warmup_steps):
        def _deriv(s, _it=it):
            ei = edges_radius_blockwise(s.pos, bc_dpos, min_radius, max_radius, block=2048)
            return model(s, ei, k=_it)
        x_probe, _ = rk4_step(x_probe, _deriv, delta_t, 'first_derivative', bc_pos)

    # --- 2. Measure pair force vs field gradient magnitudes
    ei = edges_radius_blockwise(x_probe.pos, bc_dpos, min_radius, max_radius, block=2048)
    ei_nosl = remove_self_loops(ei)[0]
    p_expand = model.p.unsqueeze(0) if model.p.dim() == 1 else model.p
    params_per_cell = p_expand[x_probe.cell_type.cpu().numpy(), :]
    src_, dst_ = ei_nosl[1], ei_nosl[0]
    msg = model.message(x_probe.pos[dst_], x_probe.pos[src_], params_per_cell[dst_])
    pair_force = scatter_aggregate(msg, dst_, x_probe.n_cells, model.aggr_type)

    res = field_params['grid_resolution']
    s_shape = [res] * dimension
    C_hat = torch.fft.rfftn(model._field_grid, s=s_shape)
    grad_grid = spectral_gradient(C_hat, res, dimension, x_probe.pos.device)
    grad_c = interp_periodic(grad_grid, x_probe.pos, res)
    pair_mag = pair_force.norm(dim=1)
    grad_mag = grad_c.norm(dim=1)

    # --- 3. Pick g0
    target = pair_mag.median().item()
    g_med = grad_mag.median().item()
    mu_chem = field_params['mu_chem']
    ratio = target / mu_chem
    g0 = g_med / (np.exp(ratio) - 1) if ratio > 1e-8 else g_med
    print(f'Warm-up done. pair median = {target:.4e}, grad median = {g_med:.4e}')
    print(f'Chosen saturation scale g0 = {g0:.4e}')

    # --- 4. Patch forward
    def _forward_sat(self, state, edge_index, has_field=False, k=0):
        edge_index = remove_self_loops(edge_index)[0]
        p = self.p.unsqueeze(0) if self.p.dim() == 1 else self.p
        parameters = p[state.cell_type.cpu().numpy(), :]

        src, dst = edge_index[1], edge_index[0]
        messages = self.message(state.pos[dst], state.pos[src], parameters[dst])
        d_pos = scatter_aggregate(messages, dst, state.n_cells, self.aggr_type)

        device = state.pos.device
        for key in ('center_0',):
            v = self.field_params.get(key)
            if torch.is_tensor(v) and v.device != device:
                self.field_params[key] = v.to(device)
        if self._field_grid is None:
            self._init_grid(device)

        steps = k - self._last_step
        if steps > 0:
            C_hat = self._advance_field(state.pos, n_steps=steps)
        else:
            s_ = [self.field_params['grid_resolution']] * self.dimension
            C_hat = torch.fft.rfftn(self._field_grid, s=s_)
        self._last_step = k

        res_ = self.field_params['grid_resolution']
        grad_grid_ = spectral_gradient(C_hat, res_, self.dimension, device)
        field_value = interp_periodic(self._field_grid, state.pos, res_)
        field_gradient = interp_periodic(grad_grid_, state.pos, res_)

        g_norm = field_gradient.norm(dim=1, keepdim=True).clamp_min(1e-12)
        g_hat = field_gradient / g_norm
        chem = self.field_params['mu_chem'] * torch.log1p(g_norm / self._chem_g0) * g_hat
        d_pos = d_pos + chem

        state.field = field_value.unsqueeze(-1)
        self.last_clean = d_pos.clone()
        if self.noise_model_level > 0:
            noise = self.noise_model_level * torch.randn_like(d_pos)
            self.last_noise = noise
            d_pos = d_pos + noise
        return d_pos

    model._chem_g0 = g0
    model.forward = types.MethodType(_forward_sat, model)
    return g0


print("Defined apply_wf_saturation(model, x_init). Call it to enable v2-style saturation.")

Defined apply_wf_saturation(model, x_init). Call it to enable v2-style saturation.


---
## Generate videos for cases 4a/4c (linear) and 4b (saturation)

Two 3-panel videos matching `generate_videos_v2.ipynb` timing (fps=30, 160 frames):
- `case4_gt_field_linear.mp4` — v1/v5 config (no saturation patch)
- `case4_gt_field_saturation.mp4` — v2 config (Weber-Fechner saturation)

In [18]:
# Reusable function: simulate + render 3-panel video.
# Timing matches generate_videos_v2.ipynb: save_every=50, n_frames=8000 -> 160 snapshots, fps=30.

SAVE_EVERY = 50      # matches frame_step in v2 notebook
FPS = 30             # matches v2 notebook
OUT_DIR = Path('../presentation/videos')
OUT_DIR.mkdir(parents=True, exist_ok=True)


def run_sim_and_make_video(model, x_init, label, output_name):
    """Run sim and save 3-panel video matching v2 notebook timing."""
    x_sim = deepcopy(x_init)
    # Reset PDE state
    model._field_grid = None
    model._last_step = -1
    model._init_grid(device='cpu')
    
    pos_history, field_history, field_grid_history, time_history = [], [], [], []
    
    for it in trange(n_frames, ncols=100, desc=f'[{label}] sim'):
        def _deriv_fn(s, _it=it):
            ei = edges_radius_blockwise(s.pos, bc_dpos, min_radius, max_radius, block=2048)
            return model(s, ei, k=_it)
        x_sim, _ = rk4_step(x_sim, _deriv_fn, delta_t, 'first_derivative', bc_pos)
        if it % SAVE_EVERY == 0:
            pos_history.append(x_sim.pos.clone().detach().numpy())
            field_history.append(x_sim.field.clone().detach().squeeze().numpy())
            time_history.append(it * delta_t)
            grid = model._field_grid.detach().cpu().numpy()
            field_grid_history.append(grid[grid_resolution // 2, :, :].copy())
    
    pos_history = np.array(pos_history)
    field_history = np.array(field_history)
    field_grid_history = np.array(field_grid_history)
    time_history = np.array(time_history)
    n_snapshots = len(time_history)
    print(f'  {n_snapshots} snapshots, field range [{field_grid_history.min():.4f}, {field_grid_history.max():.4f}]')
    
    # Render video
    vmax_field = field_grid_history.max()
    vmax_particle = max(field_history.max(), 1e-6)
    
    # Particle radius for scatter size
    particle_radius = 0.015
    fig_w_per_panel = 6.0
    pts_per_data = fig_w_per_panel * 72 * 0.65 / 1.0
    radius_pts = particle_radius * pts_per_data
    PARTICLE_S = float(np.pi * radius_pts ** 2)
    FIELD_CMAP = 'Oranges'
    
    fig = plt.figure(figsize=(18, 6))
    ax1 = fig.add_subplot(131, projection='3d')
    ax2 = fig.add_subplot(132)
    ax3 = fig.add_subplot(133)
    _seed_img = ax3.imshow(np.zeros_like(field_grid_history[0]), extent=[0, 1, 0, 1],
                            origin='lower', cmap=FIELD_CMAP, vmin=0, vmax=vmax_field)
    fig.colorbar(_seed_img, ax=ax3, fraction=0.046, pad=0.04, label='concentration c(x, t)')
    
    def update(frame_idx):
        pos = pos_history[frame_idx]
        t = time_history[frame_idx]
        fvals = field_history[frame_idx]
        field_img = field_grid_history[frame_idx]
        
        ax1.cla()
        ax1.scatter(pos[:, 0], pos[:, 1], pos[:, 2], s=PARTICLE_S, c=fvals, cmap='YlOrRd',
                   vmin=0, vmax=vmax_particle, alpha=0.7, edgecolors='none')
        ax1.set_xlim(0, 1); ax1.set_ylim(0, 1); ax1.set_zlim(0, 1)
        ax1.set_box_aspect([1, 1, 1])
        ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')
        ax1.set_title(f'3D  t={t:.2f}')
        
        ax2.cla()
        ax2.scatter(pos[:, 0], pos[:, 1], s=PARTICLE_S, c=fvals, cmap='YlOrRd',
                   vmin=0, vmax=vmax_particle, alpha=0.7, edgecolors='none')
        ax2.set_xlim(0, 1); ax2.set_ylim(0, 1); ax2.set_aspect('equal')
        ax2.set_xlabel('x'); ax2.set_ylabel('y')
        ax2.set_title('Top-down particles')
        
        ax3.cla()
        ax3.imshow(field_img, extent=[0, 1, 0, 1], origin='lower',
                   cmap=FIELD_CMAP, vmin=0, vmax=vmax_field, alpha=0.9)
        ax3.scatter(pos[:, 0], pos[:, 1], s=PARTICLE_S, c='#1f77b4',
                    alpha=0.7, edgecolors='none')
        ax3.set_xlim(0, 1); ax3.set_ylim(0, 1); ax3.set_aspect('equal')
        ax3.set_xlabel('x'); ax3.set_ylabel('y')
        ax3.set_title('Field (z=0.5) + particles')
        
        fig.suptitle(f'{label}   t={t:.2f}', fontsize=14)
    
    print(f'  Rendering {n_snapshots} frames at fps={FPS}...')
    ani = animation.FuncAnimation(fig, update, frames=n_snapshots, interval=1000/FPS)
    video_path = str(OUT_DIR / f'{output_name}.mp4')
    ani.save(video_path, writer='ffmpeg', fps=FPS, dpi=100)
    plt.close(fig)
    print(f'  -> {video_path}')


print('Ready. Call run_sim_and_make_video(model, x, label, output_name).')

Ready. Call run_sim_and_make_video(model, x, label, output_name).


In [19]:
# ---- Linear chemotaxis (v1/v5 config): no saturation patch ----
# Uses the model with only the pulsed-source patch applied.
run_sim_and_make_video(
    model, x,
    label='GT Diffusion Field — linear chemotaxis (v1/v5)',
    output_name='case4_gt_field_linear',
)

[GT Diffusion Field — linear chemotaxis (v1/v5)] sim: 100%|█████| 8000/8000 [06:17<00:00, 21.19it/s]


  160 snapshots, field range [-0.0001, 0.2320]
  Rendering 160 frames at fps=30...
  -> ../presentation/videos/case4_gt_field_linear.mp4


In [20]:
# ---- Weber-Fechner saturation (v2 config): patch model.forward then run ----
apply_wf_saturation(model, x)
run_sim_and_make_video(
    model, x,
    label='GT Diffusion Field — Weber-Fechner saturation (v2)',
    output_name='case4_gt_field_saturation',
)

Warm-up done. pair median = 5.5597e-03, grad median = 7.9888e-02
Chosen saturation scale g0 = 1.3973e+00


[GT Diffusion Field — Weber-Fechner saturation (v2)] sim: 100%|█| 8000/8000 [06:59<00:00, 19.07it/s]


  160 snapshots, field range [-0.0001, 0.1758]
  Rendering 160 frames at fps=30...
  -> ../presentation/videos/case4_gt_field_saturation.mp4


---
## v3 config forward simulation

Use `dicty_spring_force_rk4_diffusion_field_siren_grid_pde_v3.yaml`:
- 20% emitters (vs 50% in the earlier run)
- `pulse_period = 2000` steps — each emitter fires ~4 times over 8000 frames
- `pulse_duty = 0.1` — narrow ~200-step bursts
- `lambda_decay = 0.5` — faster field decay (vs 0.1 earlier)
- Linear chemotaxis (no saturation)

In [ ]:
# --- v3 config parameters ---
v3_field_params = {
    'diffusion_coeff': 0.01,
    'lambda_decay': 0.5,
    'grid_resolution': 100,
    'source_strength': 20.0,
    'center_0': torch.tensor([[0.50, 0.50, 0.50]], device=device),
    'amplitude': 0.0,
    'sigma': 0.1,
    'mu_chem': 0.1,
    'delta_t': delta_t,
    'periodic': True,
}
v3_source_fraction = 0.2
v3_pulse_period = 2000
v3_pulse_duty = 0.1

# Fresh model for v3
model_v3 = ParticleSpringForceDiffusionField(
    aggr_type='add', p=cell_params, bc_dpos=bc_dpos,
    dimension=dimension, noise_model_level=noise_level,
    field_params=v3_field_params,
)

# Fresh initial state (same seed as original)
torch.manual_seed(42)
pos_v3 = torch.rand(n_cells, dimension, device=device)
x_v3 = CellState(
    index=torch.arange(n_cells, device=device),
    pos=pos_v3,
    vel=torch.zeros(n_cells, dimension, device=device),
    cell_type=torch.zeros(n_cells, dtype=torch.long, device=device),
    field=torch.ones(n_cells, 1, device=device),
)

# Build source mask + phases for v3
torch.manual_seed(123)
n_src_v3 = max(1, int(round(v3_source_fraction * n_cells)))
perm_v3 = torch.randperm(n_cells, device=device)
source_mask_v3 = torch.zeros(n_cells, dtype=torch.bool, device=device)
source_mask_v3[perm_v3[:n_src_v3]] = True
source_phase_v3 = torch.rand(n_cells, device=device)
print(f"v3 sources: {n_src_v3}/{n_cells}, period={v3_pulse_period} steps, duty={v3_pulse_duty}")

# Patched advance_field bound to v3 pulse params
def _patched_advance_field_v3(self, pos, n_steps=1):
    dim = self.dimension
    res = self.field_params['grid_resolution']
    dt = self.field_params['delta_t']
    alpha = self.field_params.get('source_strength', 0.0)
    s = [res] * dim

    sigma = max(v3_pulse_duty, 1e-3) / 2.0
    gain = 1.0 / max(v3_pulse_duty, 1e-3)

    for i in range(n_steps):
        current_step = self._last_step + 1 + i
        if alpha > 0:
            phase = (current_step / float(v3_pulse_period) + source_phase_v3) % 1.0
            d = torch.minimum(phase, 1.0 - phase)
            pulse = torch.exp(-0.5 * (d / sigma) ** 2) * source_mask_v3.float()
            active = pulse > 1e-4
            if active.any():
                strength = alpha * dt * gain * pulse[active]
                src = _deposit_weighted(pos[active], strength, res, dim, pos.device)
                self._field_grid = self._field_grid + src

        C_hat = torch.fft.rfftn(self._field_grid, s=s)
        C_hat = C_hat * self._decay
        self._field_grid = torch.fft.irfftn(C_hat, s=s)

    C_hat = torch.fft.rfftn(self._field_grid, s=s)
    return C_hat

model_v3._advance_field = types.MethodType(_patched_advance_field_v3, model_v3)
print('Patched model_v3 with v3 pulse params (20% sources, period=2000, duty=0.1)')

In [ ]:
# Run v3 forward simulation and generate video
run_sim_and_make_video(
    model_v3, x_v3,
    label='v3: 20% pulsed sources (period=2000, duty=0.1, lambda=0.5)',
    output_name='v3_gt_forward_sim',
)